# Treino Cross-Generator — augmentation x AUC por gerador

Confirmação de verdade do scanner: o diagnóstico espectral é um *proxy*; aqui treinamos um CNN e medimos.

**Protocolo:** treina ResNet-50 no **StyleGAN (140k)** com uma receita de augmentation, e avalia no **ArtiFact por gerador** (todos os 8 são held-out — o modelo nunca os viu no treino), na **metade `test`** do split. A augmentation é a única variável.

**Receitas comparadas:**
- `raw` — sem degradação (só flip). Baseline.
- `jpeg+noise` — vencedora atual (fase 01e).
- `blur+down+noise+jpeg` — o pool âncora que o scanner recomendou (ataca espectral + resíduo, o que transfere).

Regime barato (5% do 140k, 10 épocas, 2 seeds) — resumível. A pergunta: **o pool mais rico bate o jpeg+noise no cross-generator médio?**

In [ ]:
import sys, json, time, random
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path.cwd().parent / "notebooks_140k"))
from aug_utils import RandomAugment, PathListDataset, clean_transform, train_transform, artifact_split

PROJECT_ROOT = Path.cwd().resolve().parent
_envf = PROJECT_ROOT / "data_root.env"
DATA_ROOT = Path(_envf.read_text().strip()) if _envf.exists() else PROJECT_ROOT / "data"
RAW_DIR      = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
ARTIFACT_DIR = DATA_ROOT / "raw" / "artifact_faces"
RESULTS_DIR  = PROJECT_ROOT / "artifacts" / "cross_gen_aug"
FIGS_DIR     = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True); FIGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "results.json"

IMAGE_SIZE = 224; BATCH_SIZE = 32; NUM_WORKERS = 4
NUM_EPOCHS = 10; SAMPLE_FRACTION = 0.05; LR = 1e-4; DROPOUT = 0.5
P_APPLY = 0.6
N_REAL_EVAL = 800; N_PER_GEN_EVAL = 150
SEEDS = [42, 123]

# receitas: nome -> (pool, ranges) | None = raw
RECIPES = {
    "raw": None,
    "jpeg+noise": (["jpeg", "noise"], {"jpeg": (50, 90), "noise": (0.0, 0.05)}),
    "blur+down+noise+jpeg": (["jpeg", "blur", "downscale", "noise"],
                             {"jpeg": (50, 90), "noise": (0.0, 0.05), "blur": (0.0, 1.5), "downscale": (1.0, 2.0)}),
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)
print("Receitas:", list(RECIPES), "| seeds:", SEEDS, "| runs:", len(RECIPES) * len(SEEDS))

## 1. Treino (140k StyleGAN, augmentation por receita)

In [ ]:
def make_augment(recipe):
    if RECIPES[recipe] is None:
        return None
    pool, ranges = RECIPES[recipe]
    return RandomAugment(pool, P_APPLY, ranges)

def build_train_loader(seed, recipe):
    tf = train_transform(IMAGE_SIZE, make_augment(recipe))
    ds = datasets.ImageFolder(RAW_DIR / "train", transform=tf)
    n = int(len(ds) * SAMPLE_FRACTION)
    idx = random.Random(seed).sample(range(len(ds)), n)
    sub = Subset(ds, idx)
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                      persistent_workers=NUM_WORKERS > 0, pin_memory=True)

def build_model(seed):
    torch.manual_seed(seed)
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(m.fc.in_features, 2))
    return m.to(DEVICE)

def train_model(seed, recipe):
    loader = build_train_loader(seed, recipe)
    model = build_model(seed)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    for _ in range(NUM_EPOCHS):
        model.train()
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(); loss = crit(model(imgs), lbls); loss.backward(); opt.step()
    return model

print("Funcoes de treino prontas.")

## 2. Avaliação ArtiFact por gerador (metade `test`, limpa)

Monta um conjunto fixo: reais (test) + fakes de cada gerador (test). Prediz uma vez por modelo e calcula a AUC **real-vs-cada-gerador** separadamente.

In [ ]:
def _list_by_source(folder):
    g = {}
    for p in folder.glob("*.*"):
        s = p.name.split("__")[0] if "__" in p.name else "?"
        g.setdefault(s, []).append(p)
    return {k: sorted(v) for k, v in g.items()}

_real_groups = _list_by_source(ARTIFACT_DIR / "real")
_fake_groups = _list_by_source(ARTIFACT_DIR / "fake")
GENERATORS = sorted(_fake_groups)

# metade test, disjunta do split de selecao; reais balanceados entre fontes
_rng = random.Random(42)
def _take(files, n):
    pool = artifact_split(files, which="test")
    return _rng.sample(pool, min(n, len(pool)))

real_src = sorted(_real_groups)
per = max(1, N_REAL_EVAL // len(real_src))
eval_items, eval_src = [], []
for s in real_src:
    for p in _take(_real_groups[s], per):
        eval_items.append((p, 1)); eval_src.append("real")
for g in GENERATORS:
    for p in _take(_fake_groups[g], N_PER_GEN_EVAL):
        eval_items.append((p, 0)); eval_src.append(g)

eval_loader = DataLoader(PathListDataset(eval_items, clean_transform(IMAGE_SIZE)),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
EVAL_Y = np.array([lab for _, lab in eval_items])
EVAL_SRC = np.array(eval_src)
print("eval:", (EVAL_Y == 1).sum(), "reais +", (EVAL_Y == 0).sum(), "fakes |", len(GENERATORS), "geradores")

@torch.no_grad()
def eval_per_generator(model):
    model.eval(); probs = []
    for imgs, _ in eval_loader:
        probs.append(torch.softmax(model(imgs.to(DEVICE)), 1)[:, 1].cpu().numpy())
    prob = np.concatenate(probs)
    real = EVAL_Y == 1
    out = {}
    for g in GENERATORS:
        idx = real | (EVAL_SRC == g)
        out[g] = float(roc_auc_score(EVAL_Y[idx], prob[idx]))
    out["MEAN"] = float(np.mean([out[g] for g in GENERATORS]))
    out["POOLED"] = float(roc_auc_score(EVAL_Y, prob))
    return out

print("Avaliador pronto.")

## 3. Loop dos experimentos (resumível)

Salva incrementalmente — se interromper, retoma de onde parou.

In [ ]:
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    done = {(r["recipe"], r["seed"]) for r in results}
    print(f"Retomando: {len(results)} runs feitos.")
else:
    results, done = [], set()

jobs = [(rc, sd) for rc in RECIPES for sd in SEEDS if (rc, sd) not in done]
print("Runs restantes:", len(jobs))

for recipe, seed in tqdm(jobs, desc="cross-gen aug"):
    t0 = time.time()
    model = train_model(seed, recipe)
    aucs = eval_per_generator(model)
    del model; torch.cuda.empty_cache()
    dt = time.time() - t0
    results.append({"recipe": recipe, "seed": seed, "time": round(dt, 1), **aucs})
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    tqdm.write(f"{recipe:22s} seed={seed} | MEAN={aucs['MEAN']:.3f} POOLED={aucs['POOLED']:.3f} | {dt:.0f}s")

print("Concluido.")

## 4. Resultados — receita x gerador

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

# media entre seeds por receita
order = list(RECIPES)
agg_mean = df.groupby("recipe")["MEAN"].agg(["mean", "std"]).reindex(order)
agg_pool = df.groupby("recipe")["POOLED"].mean().reindex(order)

print("AUC cross-generator MEDIA entre geradores (media +/- desvio entre seeds):\n")
for rc in order:
    s = agg_mean.loc[rc]
    print(f"  {rc:22s} MEAN={s['mean']:.4f} +/- {0.0 if np.isnan(s['std']) else s['std']:.4f}   POOLED={agg_pool[rc]:.4f}")

# tabela receita x gerador (media entre seeds)
per_gen = df.groupby("recipe")[GENERATORS].mean().reindex(order)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print("\nAUC por gerador (media entre seeds):\n")
print(per_gen.to_string())

best = agg_mean["mean"].idxmax()
print(f"\n-> Melhor receita (MEAN cross-generator): {best} = {agg_mean.loc[best,'mean']:.4f}")
winner = RECIPES.get("blur+down+noise+jpeg")
print(f"   delta do pool novo vs jpeg+noise: "
      f"{agg_mean.loc['blur+down+noise+jpeg','mean'] - agg_mean.loc['jpeg+noise','mean']:+.4f}")

In [ ]:
# grafico: barras por receita (MEAN cross-gen) com erro entre seeds
fig, ax = plt.subplots(figsize=(7, 4.5))
m = agg_mean["mean"].values
e = np.nan_to_num(agg_mean["std"].values)
ax.bar(range(len(order)), m, yerr=e, capsize=5, color=["#a0aec0", "#3182ce", "#2f855a"])
for i, v in enumerate(m):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=10)
ax.axhline(0.5, color="red", ls=":", lw=1, label="chance")
ax.set_ylabel("AUC cross-generator (media dos 8 geradores)")
ax.set_title("Augmentation x generalizacao cross-generator")
ax.set_ylim(0.5, max(0.8, m.max() + 0.05)); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS_DIR / "cross_gen_aug.png", dpi=130, bbox_inches="tight")
plt.show()

# heatmap receita x gerador
fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.imshow(per_gen.values, cmap="RdYlGn", vmin=0.5, vmax=0.9, aspect="auto")
ax.set_xticks(range(len(GENERATORS))); ax.set_xticklabels(GENERATORS, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
for i in range(len(order)):
    for j in range(len(GENERATORS)):
        ax.text(j, i, f"{per_gen.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("AUC por gerador (verde = detecta melhor)")
plt.colorbar(im, ax=ax, fraction=0.025)
plt.tight_layout()
plt.savefig(FIGS_DIR / "cross_gen_aug_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()

## 5. Leitura

- **MEAN cross-generator** é a métrica de decisão (média sobre os 8 geradores held-out). Compare `blur+down+noise+jpeg` vs `jpeg+noise` vs `raw`.
- Se o delta do pool novo vs `jpeg+noise` for **menor que o desvio entre seeds**, é empate — rode mais seeds antes de concluir.
- O **heatmap por gerador** mostra *onde* cada receita ajuda. Esperado: receitas que matam espectral ajudam os GANs (cips, projected_gan...); `diffusion_gan` deve continuar o teto duro (mal detectável).
- Próximo: se o pool rico vencer, adotar no 03/04; testar combos/encadeamento; e usar como base do classificador de geradores adversarial.